# LDA and Spectral Clustering (mining approach) Notebook

This model uses the same basic structure as the previous group, but replaces HDBSCAN with Spectral Clustering

## Workflow Outline:
We leverage two parallel pipelines, that are combined to recommend median frequencies to explore after each model has completed training and prediction.

All projects for this phase of the overall pipeline are 'line' projects.

### Frequency Mining Pipeline
* OPTIONAL: remove projects with > 26.5 measurements **CURRENTLY REMOVING**
    * Tested both options, hit rate accuracies did not increase significantly to offset 1k cluster add

* Run projects through LDA to generate topic model with $N=50$ topics
    * Currently using count vectorization of combined title and abstract with lemmatized_no_sw_text
* Group projects to max topic by taking argmax of document-topic table
* Run Spectral Clustering on each of the topics to create measurement clusters, referred to as "areas of interest"
    * Currently areas of interest are taken from min and max median frequency for each cluster generated
    * NOTE: each of the 50 Spectral Clustering models can (and probably should) be tuned individually
        * We should make sure generated clusters are not too large unless it makes sense
            * E.G. a large cluster from 700-750GHz might make sense since measurements in this range are generally sparse
    * Any cluster that spans bands is broken up into smaller clusters
        * If the remaining measurments do not meet the minimum point requirements to be a cluster they are assigned back to noise

### Band Prediction Pipeline
* OPTIONAL: remove projects with > 26.5 measurements 
* Predict band for project with Naive Bayes
    * Currently using TF-IDF vectorization of combined title and abstract with lemmatized_no_sw_text
* Choose band(s) using hard classification into one or two bands
    * We remove band 2 entirely because there are so few 
    * We do this to be able to give a final hit rate of appx. 75%
        * This shows we have a good prediction model to match projects to band
* Ultimately we will use probability vector output (not hard classification) to order mined recommendations by full band prediction

In [ ]:
import pandas as pd
import numpy as np
import pandas as pd
import plotly_express as px
from ast import literal_eval
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import HDBSCAN, OPTICS, SpectralClustering # various testing methods
from sklearn.mixture import GaussianMixture

SEED = 42

## Read data
Training and testing projects and measurements are created in the /data/Data_Ingestion.ipynb notebook

In [2]:
train_projects = pd.read_csv("C:/Users/krish/OneDrive/Desktop/Grad School Shit/Real Grad School/Spring 2025/Capstone/NRAO_ALMA_MSDS_Capstone_2024/data/raw_data/train_projects.csv")
train_projects = train_projects.set_index('project_code')
train_projects.shape

(2383, 12)

In [3]:
test_projects = pd.read_csv("C:/Users/krish/OneDrive/Desktop/Grad School Shit/Real Grad School/Spring 2025/Capstone/NRAO_ALMA_MSDS_Capstone_2024/data/raw_data/test_projects.csv")
test_projects = test_projects.set_index('project_code')
test_projects.shape

(795, 12)

In [4]:
train_measurements = pd.read_csv('C:/Users/krish/OneDrive/Desktop/Grad School Shit/Real Grad School/Spring 2025/Capstone/NRAO_ALMA_MSDS_Capstone_2024/data/raw_data/train_measurements.zip')
train_measurements = train_measurements.set_index('project_code')
train_measurements.shape

(17638, 16)

In [6]:
train_measurements.head(5)[['low_freq', 'med_freq', 'high_freq']]

,low_freq,med_freq,high_freq
project_code,,,
2016.1.01288.S,137.06,137.995,138.93
2016.1.01288.S,138.87,139.800,140.73
2016.1.01288.S,149.17,150.100,151.03
2016.1.01288.S,150.97,151.905,152.84
2016.1.01288.S,143.87,144.805,145.74


In [52]:
test_measurements = pd.read_csv('C:/Users/krish/OneDrive/Desktop/Grad School Shit/Real Grad School/Spring 2025/Capstone/NRAO_ALMA_MSDS_Capstone_2024/data/raw_data/test_measurements.zip')
test_measurements = test_measurements.set_index('project_code')
test_measurements.shape

(5844, 16)

## Read in band predictions
This data frame gives a list from least likely band to most likely band for each test project from the Band Classification part of the project

In [53]:
band_predictions = pd.read_csv('../../data/model_outputs/band_prediction.csv')
band_predictions = band_predictions.set_index('project_code')
band_predictions.head()

,band_predictions
project_code,
2016.1.00485.S,"[1, 10, 9, 5, 4, 7, 6, 8, 3]"
2017.1.00824.S,"[1, 10, 8, 5, 9, 4, 7, 3, 6]"
2015.1.01088.S,"[1, 10, 9, 8, 5, 4, 7, 6, 3]"
2013.1.00781.S,"[1, 9, 10, 8, 5, 7, 3, 4, 6]"
2016.1.00800.S,"[1, 10, 5, 4, 8, 9, 3, 6, 7]"


## Band Cutoffs from ALMA

In [54]:
band_cutoffs = [35, 51, 84, 125, 158, 211, 275, 385, 602, 787]

## LDA Model

### Create training and testing text groups to fit LDA

In [55]:
train_texts = train_projects.lemmatized_no_sw_text
test_texts = test_projects.lemmatized_no_sw_text

### LDA class

In [56]:
class LDA_Model:
    def __init__(self, N_topics=50):
        self.N_topics = N_topics
        self.countVectorizer = CountVectorizer()
        self.lda = LatentDirichletAllocation(n_components=self.N_topics, random_state=SEED)
    
    def fit(self, corpus):
        termFrequency = self.countVectorizer.fit_transform(corpus)
        self.lda.fit(termFrequency)
        return self.lda.transform(termFrequency)

    # Additional method to transform new data
    def transform(self, corpus):
        termFrequency = self.countVectorizer.transform(corpus)
        return self.lda.transform(termFrequency)

#### Initialize Model

In [57]:
lda_model = LDA_Model(N_topics=50)

#### Fit model on training set

In [58]:
train_topics = lda_model.fit(train_texts)

In [59]:
words = lda_model.countVectorizer.get_feature_names_out()

### Inspect top words for topics to see if they are salient
We can also use these later in user-facing tools for transparency

In [60]:
N = 10 #number of top words to show
topic_components = lda_model.lda.components_

for topic_idx, topic in enumerate(topic_components):
    print(f"Topic {topic_idx}:")
    # Get the indices of the top N words for this topic
    top_word_indices = topic.argsort()[-N:][::-1]
    # Print these words with their weights
    for word_idx in top_word_indices:
        print(f"{words[word_idx]} (weight: {topic[word_idx]:.2f})")
    print("\n")

Topic 0:
line (weight: 57.87)
bd (weight: 48.98)
compact (weight: 36.90)
velocity (weight: 26.63)
nucleus (weight: 24.94)
con (weight: 21.02)
obscure (weight: 17.56)
bds (weight: 17.02)
variation (weight: 16.27)
nuclei (weight: 15.03)


Topic 1:
dust (weight: 188.60)
gas (weight: 122.90)
ci (weight: 59.72)
grain (weight: 40.69)
evolution (weight: 39.17)
observation (weight: 36.21)
star (weight: 30.49)
carbon (weight: 30.10)
destruction (weight: 28.51)
study (weight: 26.06)


Topic 2:
agb (weight: 141.51)
star (weight: 127.80)
mass (weight: 107.10)
loss (weight: 81.86)
bipolar (weight: 70.32)
outflow (weight: 61.83)
jet (weight: 59.40)
nebula (weight: 53.18)
planetary (weight: 40.83)
wind (weight: 40.55)


Topic 3:
system (weight: 74.58)
debris (weight: 63.54)
belt (weight: 47.11)
planet (weight: 46.05)
disk (weight: 36.87)
collision (weight: 34.83)
structure (weight: 34.27)
disc (weight: 30.83)
observation (weight: 30.48)
scale (weight: 28.42)


Topic 4:
galaxy (weight: 212.35)
scale (

### Inspect training document-topic data frames

In [61]:
train_doc_topic = pd.DataFrame(train_topics)
train_doc_topic = train_doc_topic.set_index(train_texts.index.values)
train_doc_topic.head()

,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
2016.1.01288.S,0.000392,0.000392,0.000392,0.000392,0.000392,0.065492,0.000392,0.000392,0.000392,0.000392,...,0.000392,0.000392,0.000392,0.000392,0.000392,0.000392,0.464354,0.000392,0.000392,0.000392
2018.1.01077.S,0.000194,0.000194,0.000194,0.000194,0.000194,0.000194,0.000194,0.000194,0.000194,0.000194,...,0.000194,0.000194,0.000194,0.000194,0.000194,0.000194,0.000194,0.376111,0.000194,0.000194
2018.1.00437.S,0.000192,0.000192,0.000192,0.127172,0.000192,0.000192,0.000192,0.000192,0.000192,0.000192,...,0.000192,0.000192,0.000192,0.000192,0.000192,0.000192,0.000192,0.000192,0.000192,0.000192
2021.1.00637.S,0.000222,0.000222,0.000222,0.000222,0.000222,0.000222,0.000222,0.000222,0.000222,0.000222,...,0.000222,0.000222,0.000222,0.000222,0.000222,0.000222,0.000222,0.000222,0.000222,0.000222
2012.1.00786.S,0.000171,0.000171,0.192147,0.000171,0.000171,0.000171,0.000171,0.740628,0.000171,0.000171,...,0.000171,0.000171,0.000171,0.000171,0.000171,0.000171,0.000171,0.000171,0.000171,0.000171


### Match test data into topics

In [62]:
test_topics = lda_model.transform(test_texts)

### Inspect testing document-topic data frames

In [63]:
test_doc_topic= pd.DataFrame(test_topics.tolist())
test_doc_topic= test_doc_topic.set_index(test_texts.index.values)
test_doc_topic.head(5)

,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
2016.1.00485.S,0.000215,0.000215,0.000215,0.029035,0.000215,0.000215,0.000215,0.000215,0.000215,0.000215,...,0.000215,0.000215,0.000215,0.000215,0.000215,0.000215,0.000215,0.031655,0.000215,0.113737
2017.1.00824.S,0.000169,0.098192,0.080003,0.000169,0.000169,0.000169,0.000169,0.000169,0.000169,0.000169,...,0.000169,0.130644,0.000169,0.140330,0.000169,0.000169,0.000169,0.022981,0.000169,0.000169
2015.1.01088.S,0.188665,0.000211,0.000211,0.000211,0.017753,0.134495,0.000211,0.063180,0.063875,0.000211,...,0.000211,0.000211,0.000211,0.109437,0.000211,0.020642,0.146402,0.000211,0.000211,0.070094
2013.1.00781.S,0.000294,0.000294,0.000294,0.000294,0.000294,0.000294,0.135778,0.000294,0.000294,0.122034,...,0.000294,0.000294,0.000294,0.000294,0.000294,0.000294,0.000294,0.087903,0.000294,0.461476
2016.1.00800.S,0.317005,0.000182,0.000182,0.000182,0.000182,0.000182,0.000182,0.000182,0.000182,0.000182,...,0.000182,0.000182,0.000182,0.076003,0.000182,0.000182,0.000182,0.000182,0.052253,0.000182


### Group documents to highest matching topic

Combine project topic vector frames for convenience.
* Note you can subset this dataframe to train and test texts using `proj_topics.loc[train_texts.index]`

In [64]:
train_texts = pd.DataFrame(train_texts)
test_texts = pd.DataFrame(test_texts)
proj_topics = pd.concat([train_doc_topic, test_doc_topic])
proj_topics.head(5)

,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
2016.1.01288.S,0.000392,0.000392,0.000392,0.000392,0.000392,0.065492,0.000392,0.000392,0.000392,0.000392,...,0.000392,0.000392,0.000392,0.000392,0.000392,0.000392,0.464354,0.000392,0.000392,0.000392
2018.1.01077.S,0.000194,0.000194,0.000194,0.000194,0.000194,0.000194,0.000194,0.000194,0.000194,0.000194,...,0.000194,0.000194,0.000194,0.000194,0.000194,0.000194,0.000194,0.376111,0.000194,0.000194
2018.1.00437.S,0.000192,0.000192,0.000192,0.127172,0.000192,0.000192,0.000192,0.000192,0.000192,0.000192,...,0.000192,0.000192,0.000192,0.000192,0.000192,0.000192,0.000192,0.000192,0.000192,0.000192
2021.1.00637.S,0.000222,0.000222,0.000222,0.000222,0.000222,0.000222,0.000222,0.000222,0.000222,0.000222,...,0.000222,0.000222,0.000222,0.000222,0.000222,0.000222,0.000222,0.000222,0.000222,0.000222
2012.1.00786.S,0.000171,0.000171,0.192147,0.000171,0.000171,0.000171,0.000171,0.740628,0.000171,0.000171,...,0.000171,0.000171,0.000171,0.000171,0.000171,0.000171,0.000171,0.000171,0.000171,0.000171


### Take highest matching topic for each project

In [65]:
proj_topics['max_topic'] = proj_topics.apply(lambda x: x.argmax(), axis=1)

### Create data frame with project id and max topic

In [66]:
proj_max_topic = proj_topics['max_topic'].to_frame()
proj_max_topic.head()

,max_topic
2016.1.01288.S,46
2018.1.01077.S,37
2018.1.00437.S,17
2021.1.00637.S,13
2012.1.00786.S,7


In [67]:
proj_max_topic.max_topic.value_counts().to_frame().sort_index()

,count
max_topic,
0,23
1,29
2,42
3,24
4,47
5,37
6,50
7,25
8,28


### Inspect some topic stats

In [68]:
proj_max_topic.value_counts().describe()

count     50.0000
mean      63.5600
std       64.2737
min        7.0000
25%       24.2500
50%       33.5000
75%       92.7500
max      293.0000
Name: count, dtype: float64

There are a few topics that match to a large number of documents. Perhaps we need a better topic model or to group documents by project_topic vector similarity.

### Eyeball comparison of documents by max topic
This requires looking at the online explorer since printing out abstracts in here gets messy.

In [69]:
proj_max_topic[proj_max_topic.max_topic == 3].head()

,max_topic
2022.1.00793.S,3
2015.1.01260.S,3
2017.1.00167.S,3
2019.1.01443.T,3
2015.1.00032.S,3


### Add `max_topic` to `measurements` frame to be able to group measurements by max topic

In [70]:
train_measurements = pd.merge(train_measurements, proj_max_topic, left_index=True, right_index=True)

### Generate test projects measurements
This will be useful for calculating hit rates to evaluate model performance.

**NOTE!!!**
You should not sort these, however tempting. We need to preserve the relationships of the entries to not lose measurement information.

In [71]:
test_proj_meas = test_measurements.loc[test_texts.index]
test_proj_meas = test_proj_meas.groupby(test_proj_meas.index)\
    .agg({
        'low_freq': lambda x: round(x, 4).tolist(),
        'high_freq': lambda x: round(x, 4).tolist(),
        'med_freq': lambda x: round(x, 4).tolist(),
        'diff_freq': lambda x: round(x, 4).tolist()
    })
test_proj_meas.head()

,low_freq,high_freq,med_freq,diff_freq
project_code,,,,
2011.0.00010.S,"[90.38, 90.7, 91.69, 92.89, 217.59, 218.67, 21...","[90.62, 90.93, 91.92, 93.12, 218.53, 219.6, 21...","[90.5, 90.815, 91.805, 93.005, 218.06, 219.135...","[0.24, 0.23, 0.23, 0.23, 0.94, 0.93, 0.94, 0.9..."
2011.0.00064.S,"[288.96, 290.79, 300.84, 302.71, 288.94, 290.7...","[290.84, 292.67, 302.71, 304.59, 290.82, 292.6...","[289.9, 291.73, 301.775, 303.65, 289.88, 291.7...","[1.88, 1.88, 1.87, 1.88, 1.88, 1.87, 1.88, 1.87]"
2011.0.00121.S,"[319.07, 320.48, 319.83, 319.36, 319.71, 316.59]","[320.94, 322.35, 321.71, 321.24, 321.58, 318.47]","[320.005, 321.415, 320.77, 320.3, 320.645, 317...","[1.87, 1.87, 1.88, 1.88, 1.87, 1.88]"
2011.0.00136.S,"[335.29, 335.98, 345.67, 346.47]","[335.52, 336.22, 345.91, 346.7]","[335.405, 336.1, 345.79, 346.585]","[0.23, 0.24, 0.24, 0.23]"
2011.0.00199.S,"[639.15, 645.41, 657.7, 661.7, 320.98, 322.12,...","[640.11, 646.37, 658.66, 662.66, 321.46, 322.6...","[639.63, 645.89, 658.18, 662.18, 321.22, 322.3...","[0.96, 0.96, 0.96, 0.96, 0.48, 0.48, 0.49, 0.48]"


### Generate train topic measurements
We will use these to engineer 'areas of interest' among topics using Spectral Clustering

**NOTE!!!**
You should not sort these, however tempting. We need to preserve the relationships of the entries to not lose measurement information.

In [72]:
train_measurements.head()

,project_title,project_abstract,fs_type,low_freq,high_freq,science_category,science_keyword,band,target,diff_freq,med_freq,raw_text,standardized_text,no_sw_text,lemmatized_sw_text,lemmatized_no_sw_text,max_topic
project_code,,,,,,,,,,,,,,,,,
2016.1.01288.S,The molecular gas properties of radio-AGN at z...,Energetic feedback from AGN is believed to pla...,line,137.06,138.93,Active galaxies,"Outflows, jets, feedback",4.0,1,1.87,137.995,The molecular gas properties of radio-AGN at z...,the molecular gas properties of radio agn at z...,molecular gas properties radio agn z energetic...,the molecular gas property of radio agn at z e...,molecular gas property radio agn z energetic f...,46
2016.1.01288.S,The molecular gas properties of radio-AGN at z...,Energetic feedback from AGN is believed to pla...,line,138.87,140.73,Active galaxies,"Outflows, jets, feedback",4.0,1,1.86,139.800,The molecular gas properties of radio-AGN at z...,the molecular gas properties of radio agn at z...,molecular gas properties radio agn z energetic...,the molecular gas property of radio agn at z e...,molecular gas property radio agn z energetic f...,46
2016.1.01288.S,The molecular gas properties of radio-AGN at z...,Energetic feedback from AGN is believed to pla...,line,149.17,151.03,Active galaxies,"Outflows, jets, feedback",4.0,1,1.86,150.100,The molecular gas properties of radio-AGN at z...,the molecular gas properties of radio agn at z...,molecular gas properties radio agn z energetic...,the molecular gas property of radio agn at z e...,molecular gas property radio agn z energetic f...,46
2016.1.01288.S,The molecular gas properties of radio-AGN at z...,Energetic feedback from AGN is believed to pla...,line,150.97,152.84,Active galaxies,"Outflows, jets, feedback",4.0,1,1.87,151.905,The molecular gas properties of radio-AGN at z...,the molecular gas properties of radio agn at z...,molecular gas properties radio agn z energetic...,the molecular gas property of radio agn at z e...,molecular gas property radio agn z energetic f...,46
2016.1.01288.S,The molecular gas properties of radio-AGN at z...,Energetic feedback from AGN is believed to pla...,line,143.87,145.74,Active galaxies,"Outflows, jets, feedback",4.0,1,1.87,144.805,The molecular gas properties of radio-AGN at z...,the molecular gas properties of radio agn at z...,molecular gas properties radio agn z energetic...,the molecular gas property of radio agn at z e...,molecular gas property radio agn z energetic f...,46


In [73]:
train_topic_freqs = train_measurements.loc[train_texts.index]\
    .reset_index()\
    .groupby('max_topic')\
    .agg({
        'project_code': lambda x: x.tolist(), 
        'low_freq': lambda x: round(x, 4).tolist(),
        'high_freq': lambda x: round(x, 4).tolist(),
        'med_freq': lambda x: round(x, 4).tolist(),
        'diff_freq': lambda x: round(x, 4).tolist(),
        'band': lambda x: x.astype('int64').tolist()
    })
train_topic_freqs.head()

,project_code,low_freq,high_freq,med_freq,diff_freq,band
max_topic,,,,,,
0,"[2017.1.00598.S, 2017.1.00598.S, 2017.1.00598....","[257.18, 259.1, 260.42, 262.37, 260.84, 262.79...","[259.04, 260.97, 262.29, 264.24, 262.71, 264.6...","[258.11, 260.035, 261.355, 263.305, 261.775, 2...","[1.86, 1.87, 1.87, 1.87, 1.87, 1.87, 1.87, 1.8...","[6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 5, 5, ..."
1,"[2018.1.00341.S, 2018.1.00341.S, 2018.1.00341....","[342.03, 343.99, 354.03, 355.91, 478.54, 480.5...","[343.9, 345.86, 355.9, 357.79, 480.54, 482.54,...","[342.965, 344.925, 354.965, 356.85, 479.54, 48...","[1.87, 1.87, 1.87, 1.88, 2.0, 2.0, 2.0, 0.25, ...","[7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, ..."
2,"[2017.1.00595.S, 2017.1.00595.S, 2017.1.00595....","[330.25, 331.25, 342.52, 345.1, 215.39, 217.28...","[331.25, 333.25, 344.52, 346.1, 217.39, 219.28...","[330.75, 332.25, 343.52, 345.6, 216.39, 218.28...","[1.0, 2.0, 2.0, 1.0, 2.0, 2.0, 1.0, 2.0, 2.0, ...","[7, 7, 7, 7, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, ..."
3,"[2022.1.00793.S, 2022.1.00793.S, 2022.1.00793....","[327.94, 329.94, 339.99, 342.24, 344.24, 354.2...","[329.82, 331.82, 341.86, 344.12, 346.12, 356.1...","[328.88, 330.88, 340.925, 343.18, 345.18, 355....","[1.88, 1.88, 1.87, 1.88, 1.88, 1.87, 1.87, 1.8...","[7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 7, 7, 6, 6, 6, ..."
4,"[2017.1.00707.S, 2017.1.00707.S, 2017.1.00707....","[216.05, 217.07, 217.21, 218.19, 218.45, 219.5...","[216.17, 217.13, 217.27, 218.25, 218.5, 219.59...","[216.11, 217.1, 217.24, 218.22, 218.475, 219.5...","[0.12, 0.06, 0.06, 0.06, 0.05, 0.06, 0.06, 0.1...","[6, 6, 6, 6, 6, 6, 6, 6, 6, 3, 3, 3, 3, 3, 3, ..."


In [74]:
len(train_topic_freqs.loc[10].project_code)

1234

## Cluster cleaning function

How do we handle > 2 band overlaps?

* Currently we just roll with it.
    
* Should call out which topic, and cluster

* Alternatively, throw error <- this is hard because it completely stops any training

In [ ]:
# Code to check for clusters that span at least two bands
def cluster_cleaning(topic_meas_df, min_cluster_size):
    dummy_label = 100000  # Used to make new labels and ensure we're adding new clusters. Cluster labels will be reset eventually

    for clst in np.unique(topic_meas_df.cluster_label):
        if clst != -1:
            # Subset topic measurement dataframe to current cluster
            clst_subset = topic_meas_df[topic_meas_df.cluster_label == clst].sort_values('med_freq')
            
            # Extract band information for cluster subset
            band = np.unique(clst_subset.band)

            # If there are multiple bands in measurements for cluster we want to break them up
            if band.size > 1:
                # Loop over bands in cluster and make a dataframe for each band
                bnd_dict = [clst_subset[clst_subset.band == bnd] for bnd in band]

                # Loop over cluster-band frames
                for df in bnd_dict:
                # Check if number of measurements in this band and cluster is > min_cluster_size
                # If it is less, simply assign those measurements back to noise, since we don't want too small clusters
                # Otherwise, there are enough measurments to keeping "this" part of the cluster, so make a new cluster for it 
                    if df.shape[0] < min_cluster_size:
                        topic_meas_df.loc[df.index, 'cluster_label'] = -1
                    else:
                        topic_meas_df.loc[df.index, 'cluster_label'] = dummy_label
                        dummy_label += 1
                        
    # Re-label clusters to be a continuous range from -1 to N
    new_labels = [n-1 for n in range(len(np.unique(topic_meas_df.cluster_label)))]
    label_counter = 0   # Used to increment through new_labels
    
    # Loop over cluster labels and update them
    for reclust in np.unique(topic_meas_df.cluster_label):
        # Set cluster label to new_labels
        topic_meas_df.loc[topic_meas_df.cluster_label == reclust, 'cluster_label'] = new_labels[label_counter]
        label_counter += 1

## Spectral Clustering Train/Test Code
### Loop over topics and find accuracy measurements

In [ ]:
# Quick test
db = SpectralClustering(n_clusters=10).fit(list(zip(train_topic_freqs.loc[0].med_freq)))

db.labels_

c:\Users\krish\miniconda3\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:329: UserWarning:

Graph is not fully connected, spectral embedding may not work as expected.



array([0, 0, 0, 0, 0, 0, 5, 5, 0, 0, 0, 0, 2, 2, 0, 2, 4, 4, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 9,
       0, 7, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 8, 9, 0, 7, 9, 7, 0, 0, 0, 6, 6, 6,
       6, 0, 3, 3, 0, 0, 5, 8, 0, 0, 5, 5, 0, 0, 0])

In [ ]:
# Spectral Clustering Within all 50 topics

band_prediction_limit = 0               # Number of top band predictions to include. 0 to include all
test_project_hits = 0                   # Hits for all projects if at least one measurement is matched
test_project_meas_hit_rate = []         # List of hit rates by project
topic_cluster_widths = []               # List of cluster widths by topic to ensure generated clusters are not too wide (list of lists)
total_num_clusters = 0                  # List of number of clusters for each topic
topic_cluster_stat_list = []            # List of dataframes with clusters by topic. Used to make a main topic-cluster data frame later
topic_measurement_stat_list = []        # List of dataframes with measurements by topic. Used to make a main topic-measurement data frame later
test_project_hit_list = []              # List of test project hit rates for each cluster

# Loop over topics
for tpc in set(proj_max_topic.max_topic.values):
    # Fit Spectral Clustering for each topic
    # Note that these can be parameterized for each of the topics generated
    db = SpectralClustering()\
        .fit(list(zip(train_topic_freqs.loc[tpc].med_freq)))
    
    # Get labels from Spectral
    labels = db.labels_

    # Create topic-measurement dataframe
    # This has all of the measurements for all of the projects in an LDA topic
    topic_measurement = pd.DataFrame.from_dict({'med_freq':train_topic_freqs.loc[tpc].med_freq,
                                            'band':train_topic_freqs.loc[tpc].band,
                                            'project_code':train_topic_freqs.loc[tpc].project_code,
                                            'cluster_label':labels})
    
    # Format topic_measurement
    topic_measurement = pd.concat({tpc: topic_measurement}, names=['topic'])
    topic_measurement.index.names = ['topic', 'measurement']

    # Append topic_cluster to topic_cluster_stats for analysis later
    topic_measurement_stat_list.append(topic_measurement)
    
    # Clean clusters in topic_measurment, breaking up clusters that span more than one band
    cluster_cleaning(topic_measurement, 5)

    # Generate topic_cluster data frame for this topic
    # Clusters are defined by the minimum and maximum median_freq for all labeled measurements
    topic_cluster = topic_measurement.groupby('cluster_label').agg(
        mean_freq=('med_freq', 'mean'),
        min_freq=('med_freq', 'min'),
        max_freq=('med_freq', 'max'),
        count_freq=('med_freq', 'count'),
        count_proj=('project_code', 'nunique'),
        band_min=('band', 'min'),
        band_max=('band', 'max'),
        band_mode=('band', 'mean')
    )

    # Stat aggregation
    n_projects = topic_cluster.count_proj.sum()
    n_measurements = topic_cluster.count_freq.sum()
    if -1 in topic_cluster.index:
        n_noise = topic_cluster.loc[-1].count_freq
        noise_proportion = n_noise/n_measurements
    else:
        n_noise = 0
        noise_proportion = 0
    signal_proportion = 1-noise_proportion


    # Stat callouts
    print(f'Spectral Results for topic {tpc}')
    print(f'Number of projects in topic: {n_projects}')
    print(f'Total number of measurements: {n_measurements}')
    print(f'Estimated number of noise measurements: {n_noise}')
    print(f'Noise proportion: {round(noise_proportion, 3)}')
    print(f'Signal proportion: {round(signal_proportion, 3)}')

    # Sort index
    topic_cluster = topic_cluster.sort_index()

    # Add width for cleaned clusters
    topic_cluster['width'] = topic_cluster.max_freq - topic_cluster.min_freq

    # Add topic index to topic_cluster
    topic_cluster = pd.concat({tpc: topic_cluster}, names=['topic'])
    topic_cluster.index.names = ['topic', 'cluster']

    # Append topic_cluster to topic_cluster_stats for analysis later
    topic_cluster_stat_list.append(topic_cluster)

    # Testing loop
    # Loop over generated clusters and print cluster stats
    # Initialize list of cluster widths
    # Code to check for clusters that span at least two bands
    cluster_widths = []

    # If there are any measurements in that topic, print stats
    # This if statement is to avoid errors if topics only have noise and no clusters
    if topic_cluster.shape[0] != 0:
        for clst in topic_cluster.index:
            min_freq, max_freq = topic_cluster.loc[clst].min_freq, topic_cluster.loc[clst].max_freq
            cluster_widths.append(max_freq - min_freq)
            total_num_clusters += 1
            if (topic_cluster.loc[clst].band_min != topic_cluster.loc[clst].band_max):
                    print("BAND OVERLAP")
        print('')
        print(f'Topic {tpc} cluster width stats:')
        print(np.round(pd.Series(cluster_widths).describe(), 4))
    else: print('No clusters for this topic')

    # Print cluster data frame with relevant columns for tuning
    print('')
    print(f'Cluster data frame for topic {tpc}')
    with pd.option_context('display.max_rows', None):
        print(topic_cluster[['min_freq', 'max_freq', 'count_freq', 'band_mode', 'width']]\
          .sort_values(['width', 'min_freq'], ascending=False))

    # Get a list of test project codes
    tps = proj_max_topic.loc[test_texts.index].query(f'max_topic == {tpc}')

    #Begin test projects
    print('')
    # print('Begin tests')

    # Loop over test projects
    for tp in tps.index:
        tp_hr = 0   # Hit rate for this specific project
        # Loop over measurements in test project

        # Subset `topic_cluster` to be the top two predicted bands for each project
        topic_cluster_subset = topic_cluster[topic_cluster.band_mode.isin(literal_eval(band_predictions.loc[tp].band_predictions)[-band_prediction_limit:])]
        if topic_cluster.shape[0] != 0:
            print(f'Ratio of recommended clusters to total clusters: {topic_cluster_subset.shape[0]/topic_cluster.shape[0]}')
        else: print(f'No clusters for topic {tpc}')

        for meas in test_proj_meas.loc[tp].med_freq:
            # Loop over clusters in topic
            # Note we have a multi index so we want to get to the 'cluster' index, being level 1
            for clust in topic_cluster_subset.index.get_level_values(level=1):
                # Skip noise
                if clust != -1:
                    lower_bound = round(topic_cluster_subset.loc[tpc, clust].min_freq, 3)
                    upper_bound = round(topic_cluster_subset.loc[tpc, clust].max_freq, 3)
                    if ((meas >= lower_bound) and (meas <= upper_bound)):
                        tp_hr += 1
                        break
        test_project_meas_hit_rate.append(round(tp_hr/len(list(test_proj_meas.loc[tp].med_freq)), 3))
        test_project_hit_list.append(round(tp_hr/len(list(test_proj_meas.loc[tp].med_freq)), 3))
        # Stats for individual test projects
        print(f'Number of measurements: {len(test_proj_meas.loc[tp].med_freq)}')
        print(f'Hits: {tp_hr}')
        print(f'Hit rate: {round(tp_hr/len(list(test_proj_meas.loc[tp].med_freq)), 3)}')
        print('')

        # Increment test_project_hits if at least one measurement in the project matched
        if (tp_hr > 0):
            test_project_hits +=1
    print('=========================================\n')

print(f'Total number of clusters across topics: {total_num_clusters}')
print(f'Number of test projects with at least one measurement match: {test_project_hits}')
print(f'Ratio of test project hits to number of test projects: {round(test_project_hits/test_texts.shape[0], 4)}')
print(f'Average hit rate per project: {round(np.mean(test_project_hit_list), 4)}')
print(f'Standard Deviation of hit rate per project: {round(np.std(test_project_hit_list), 4)}')

c:\Users\krish\miniconda3\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:329: UserWarning:

Graph is not fully connected, spectral embedding may not work as expected.

c:\Users\krish\miniconda3\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:329: UserWarning:

Graph is not fully connected, spectral embedding may not work as expected.



Spectral Results for topic 0
Number of projects in topic: 26
Total number of measurements: 125
Estimated number of noise measurements: 4.0
Noise proportion: 0.032
Signal proportion: 0.968

Topic 0 cluster width stats:
count    11.0000
mean     11.6059
std      15.1723
min       1.2550
25%       2.7625
50%       5.9850
75%      13.0075
max      52.2800
dtype: float64

Cluster data frame for topic 0
               min_freq  max_freq  count_freq  band_mode   width
topic cluster                                                   
0      8        216.860   269.140          51        6.0  52.280
       7         86.845   110.895          24        3.0  24.050
       9        450.495   466.225          19        8.0  15.730
       5        244.245   254.530          11        6.0  10.285
       1        175.765   182.000           3        5.0   6.235
       0        345.945   351.930           4        7.0   5.985
      -1        338.130   342.875           4        7.0   4.745
       2      

c:\Users\krish\miniconda3\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:329: UserWarning:

Graph is not fully connected, spectral embedding may not work as expected.

c:\Users\krish\miniconda3\Lib\site-packages\sklearn\base.py:1389: ConvergenceWarning:

Number of distinct clusters (4) found smaller than n_clusters (8). Possibly due to duplicate points in X.



Spectral Results for topic 2
Number of projects in topic: 64
Total number of measurements: 260
Estimated number of noise measurements: 8.0
Noise proportion: 0.031
Signal proportion: 0.969
BAND OVERLAP

Topic 2 cluster width stats:
count     10.0000
mean      31.4185
std       61.3406
min        0.0000
25%        4.2887
50%        6.4850
75%       26.4550
max      201.4950
dtype: float64

Cluster data frame for topic 2
               min_freq  max_freq  count_freq  band_mode    width
topic cluster                                                    
2     -1        489.790   691.285           8        8.5  201.495
       3        293.875   337.100          51        7.0   43.225
       2        214.965   245.815          41        6.0   30.850
       7         93.250   106.520          25        3.0   13.270
       1         85.355    92.270          19        3.0    6.915
       8        230.290   236.345          41        6.0    6.055
       4        109.765   115.280          17     

c:\Users\krish\miniconda3\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:329: UserWarning:

Graph is not fully connected, spectral embedding may not work as expected.

c:\Users\krish\miniconda3\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:329: UserWarning:

Graph is not fully connected, spectral embedding may not work as expected.



...

Spectral Results for topic 47
Number of projects in topic: 99
Total number of measurements: 558
Estimated number of noise measurements: 2.0
Noise proportion: 0.004
Signal proportion: 0.996

Topic 47 cluster width stats:
count    13.0000
mean     21.1181
std      26.6883
min       1.7600
25%       1.8750
50%       7.4550
75%      29.0650
max      88.6350
dtype: float64

Cluster data frame for topic 47
               min_freq  max_freq  count_freq  band_mode   width
topic cluster                                                   
47     10       279.510   368.145         100        7.0  88.635
       9        212.045   272.355         146        6.0  60.310
       6         84.080   115.270         232        3.0  31.190
       7        126.100   155.165          29        4.0  29.065
       11       660.205   687.590          23        9.0  27.385
       8        183.305   195.950           6        5.0  12.645
       5        414.635   422.090           6        8.0   7.455
       2   

c:\Users\krish\miniconda3\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:329: UserWarning:

Graph is not fully connected, spectral embedding may not work as expected.

c:\Users\krish\miniconda3\Lib\site-packages\sklearn\base.py:1389: ConvergenceWarning:

Number of distinct clusters (2) found smaller than n_clusters (8). Possibly due to duplicate points in X.



Ratio of recommended clusters to total clusters: 1.0
Number of measurements: 4
Hits: 4
Hit rate: 1.0

Ratio of recommended clusters to total clusters: 1.0
Number of measurements: 2
Hits: 2
Hit rate: 1.0

Ratio of recommended clusters to total clusters: 1.0
Number of measurements: 4
Hits: 4
Hit rate: 1.0

Ratio of recommended clusters to total clusters: 1.0
Number of measurements: 22
Hits: 22
Hit rate: 1.0

Ratio of recommended clusters to total clusters: 1.0
Number of measurements: 10
Hits: 10
Hit rate: 1.0

Ratio of recommended clusters to total clusters: 1.0
Number of measurements: 1
Hits: 1
Hit rate: 1.0

Ratio of recommended clusters to total clusters: 1.0
Number of measurements: 5
Hits: 5
Hit rate: 1.0

Ratio of recommended clusters to total clusters: 1.0
Number of measurements: 3
Hits: 3
Hit rate: 1.0

Ratio of recommended clusters to total clusters: 1.0
Number of measurements: 4
Hits: 4
Hit rate: 1.0

Ratio of recommended clusters to total clusters: 1.0
Number of measurements: 8

c:\Users\krish\miniconda3\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:329: UserWarning:

Graph is not fully connected, spectral embedding may not work as expected.



Spectral Results for topic 49
Number of projects in topic: 241
Total number of measurements: 1559
Estimated number of noise measurements: 4.0
Noise proportion: 0.003
Signal proportion: 0.997

Topic 49 cluster width stats:
count     15.0000
mean      29.4913
std       34.7483
min        0.1700
25%        5.4175
50%        8.0300
75%       41.5725
max      106.2550
dtype: float64

Cluster data frame for topic 49
               min_freq  max_freq  count_freq  band_mode    width
topic cluster                                                    
49     12       391.245   497.500         180        8.0  106.255
       11       276.105   371.755         261        7.0   95.650
       10       211.975   273.955         535        6.0   61.980
       9        159.265   209.755          35        5.0   50.490
       8        129.335   161.990          84        4.0   32.655
       13       657.640   688.520          26        9.0   30.880
       7         84.980   114.985         409        3.0  

### Build full topic-measurement data frame

In [78]:
topic_measurement_frame = pd.concat(topic_measurement_stat_list)

In [79]:
topic_measurement_frame.loc[0]

,med_freq,band,project_code,cluster_label
measurement,,,,
0,258.110,6,2017.1.00598.S,8
1,260.035,6,2017.1.00598.S,8
2,261.355,6,2017.1.00598.S,8
3,263.305,6,2017.1.00598.S,8
4,261.775,6,2017.1.00598.S,8
...,...,...,...,...
120,247.295,6,2017.1.00113.S,5
121,249.065,6,2017.1.00113.S,5
122,261.235,6,2017.1.00113.S,8


### Visualization of Spectral Clustering on Topics
Choose a topic in `inspect_topic` and let it rip! It uses the `topic_measurement_frame` to generate the images. There's a little bit of extra processing though, so we create a helper dataframe, `inspect_topic_frame` to make sure everything runs smoothly.

* This should probably become a function, at least the plotting part
* Maybe check out plotly "strip" charts

In [80]:
inspect_topic = 0
inspect_topic_frame = topic_measurement_frame.loc[inspect_topic]

inspect_topic_frame = inspect_topic_frame.sort_values('cluster_label', ascending=False)
inspect_topic_frame.cluster_label = inspect_topic_frame.cluster_label.astype('str')

# Add noise binary column for plot symbol
inspect_topic_frame['noise'] = np.where(inspect_topic_frame.cluster_label == '-1', 1, 0)

# Noise and Signal
itf_noise = inspect_topic_frame.noise.sum()
itf_signal = inspect_topic_frame.shape[0] - itf_noise

# Set symbols for plot
# We set all points to be 'circle' using the px number 0, and then change noise to 'x'
symbols = list(np.zeros(np.unique(inspect_topic_frame.cluster_label).shape[0], 'int'))
symbols[-1] = 'x'

# Create plot
fig = px.scatter(inspect_topic_frame,
                 x='med_freq',
                 y='cluster_label',
                 color='cluster_label',
                 symbol='cluster_label',
                 symbol_sequence=symbols,
                 title=f"Spectral Generated Clusters for Topic {inspect_topic} <br><sup>{itf_signal} Clustered Measurements with {itf_noise} Noise Measurements</sup>",
                 labels={
                     'med_freq':'Median Frequency (GHz)',
                     'index':'Index',
                     'cluster_label':'Cluster Label'
                 })
fig.update_traces(marker={'size': 15, 'opacity':0.5})

fig.show()

In [81]:
topic_measurement_frame.loc[25]

,med_freq,band,project_code,cluster_label
measurement,,,,
0,140.935,4,2015.1.00117.S,7
1,142.815,4,2015.1.00117.S,7
2,152.935,4,2015.1.00117.S,1
3,154.815,4,2015.1.00117.S,1
4,96.165,3,2015.1.00117.S,6
...,...,...,...,...
134,159.115,4,2017.1.00321.S,1
135,132.550,4,2017.1.00321.S,7
136,134.415,4,2017.1.00321.S,7


### Build full topic-cluster data frame

## **THIS IS A VERY IMPORTANT DATAFRAME IT THIS IS THE CORE RESULT OF THE MINING APPROACH!!!!!!**

This data frame holds all of the cluster info for each of the generated topics

* Pretty much all of the cluster stats in the code cell above can be derived from this

In [82]:
topic_cluster_stats = pd.concat(topic_cluster_stat_list)

In [83]:
topic_cluster_stats.loc[0].sort_values('count_proj', ascending=False)

,mean_freq,min_freq,max_freq,count_freq,count_proj,band_min,band_max,band_mode,width
cluster,,,,,,,,,
8,243.482647,216.860,269.140,51,9,6,6,6.0,52.280
5,250.560909,244.245,254.530,11,4,6,6,6.0,10.285
7,102.378125,86.845,110.895,24,3,3,3,3.0,24.050
-1,340.390000,338.130,342.875,4,2,7,7,7.0,4.745
0,348.951250,345.945,351.930,4,2,7,7,7.0,5.985
1,178.390000,175.765,182.000,3,1,5,5,5.0,6.235
2,165.958333,164.165,168.005,3,1,5,5,5.0,3.840
3,329.952500,329.325,330.580,2,1,7,7,7.0,1.255
4,145.757500,144.915,146.600,2,1,4,4,4.0,1.685


In [84]:
topic_cluster_stats.sample(10)

,,mean_freq,min_freq,max_freq,count_freq,count_proj,band_min,band_max,band_mode,width
topic,cluster,,,,,,,,,
28,2,252.143750,250.690,253.665,4,1,6,6,6.0,2.975
1,2,185.745000,184.845,186.645,2,1,5,5,5.0,1.800
14,6,109.058125,107.895,110.115,8,4,3,3,3.0,2.220
36,-1,169.352500,164.265,174.440,2,1,5,5,5.0,10.175
49,4,173.712857,169.800,177.830,7,4,5,5,5.0,8.030
26,7,141.179375,133.795,146.970,8,1,4,4,4.0,13.175
48,-1,849.531250,840.495,860.450,4,1,10,10,10.0,19.955
43,11,474.434167,455.985,492.155,12,3,8,8,8.0,36.170
47,8,188.473333,183.305,195.950,6,2,5,5,5.0,12.645


### Compute cluster signal to noise proportions

In [85]:
sn_list = []
for clst in np.unique(topic_cluster_stats.index.get_level_values(0)):
    clst_sig = np.sum(topic_cluster_stats.loc[topic_cluster_stats.index.get_level_values(1) != -1].loc[clst].count_freq)
    clst_noise = np.sum(topic_cluster_stats.loc[topic_cluster_stats.index.get_level_values(1) == -1].loc[clst].count_freq)
    sn_list.append({'signal':clst_sig, 'noise':clst_noise})

In [86]:
signal_noise_frame = pd.DataFrame(sn_list)
signal_noise_frame.index.name = 'cluster'
signal_noise_frame['signal_prop'] = (signal_noise_frame.signal)/(signal_noise_frame.signal + signal_noise_frame.noise)
signal_noise_frame['noise_prop'] = (signal_noise_frame.noise)/(signal_noise_frame.signal + signal_noise_frame.noise)
signal_noise_frame

,signal,noise,signal_prop,noise_prop
cluster,,,,
0,121,4,0.968000,0.032000
1,203,6,0.971292,0.028708
2,252,8,0.969231,0.030769
3,72,2,0.972973,0.027027
4,289,4,0.986348,0.013652
5,286,5,0.982818,0.017182
6,264,90,0.745763,0.254237
7,142,1,0.993007,0.006993
8,174,2,0.988636,0.011364


In [87]:
signal_noise_frame.describe()

,signal,noise,signal_prop,noise_prop
count,50.000000,50.000000,50.000000,50.000000
mean,346.540000,6.220000,0.974408,0.025592
std,331.312801,12.805117,0.037630,0.037630
min,33.000000,1.000000,0.745763,0.001193
25%,139.750000,2.000000,0.970531,0.008168
50%,232.000000,4.000000,0.983192,0.016808
75%,391.750000,4.750000,0.991832,0.029469
max,1555.000000,90.000000,0.998807,0.254237


## Inspect topic-clusters

In [88]:
topic_cluster_stats.describe()

,mean_freq,min_freq,max_freq,count_freq,count_proj,band_min,band_max,band_mode,width
count,533.000000,533.000000,533.00000,533.000000,533.000000,533.000000,533.000000,533.000000,533.000000
mean,299.837739,289.142458,311.95591,33.091932,6.011257,5.996248,6.067542,6.023581,22.813452
std,185.099164,185.847621,186.90106,69.598803,10.620941,1.967782,1.970443,1.954946,50.257074
min,37.103333,36.080000,38.02000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000
25%,155.229375,150.025000,161.99000,3.000000,1.000000,4.000000,4.000000,4.000000,2.000000
50%,248.341341,239.645000,267.85500,7.000000,2.000000,6.000000,6.000000,6.000000,6.905000
75%,353.716250,345.815000,366.69000,28.000000,5.000000,7.000000,7.000000,7.000000,28.695000
max,903.982500,901.170000,906.79500,535.000000,83.000000,10.000000,10.000000,10.000000,504.010000


In [89]:
topic_cluster_stats.width.describe()

count    533.000000
mean      22.813452
std       50.257074
min        0.000000
25%        2.000000
50%        6.905000
75%       28.695000
max      504.010000
Name: width, dtype: float64

In [90]:
topic_cluster_stats.query('width > 10 and cluster != -1')\
    .sort_values(['count_freq', 'width'], ascending=False)\
    .head(50)


mean_freq  min_freq  max_freq  count_freq  count_proj  \
topic cluster                                                           
49    10       242.512757   211.975   273.955         535          77   
43    9        228.312557   213.595   267.645         483          83   
10    6         98.277436    85.010   115.270         470          68   
      8        225.221812   212.190   247.215         436          67   
49    7         99.135147    84.980   114.985         409          57   
12    0        102.837201    85.120   115.275         343          45   
24    6         94.008548    85.450   115.320         341          48   
43    10       333.229265   277.985   364.270         340          63   
46    9        231.509540   212.040   269.000         337          55   
21    8        223.376830   213.010   236.455         317          41   
41    8        225.485318   213.345   261.240         283          37   
24    8        224.519065   215.200   234.975         262          33   
49    11       316.728180   276.105   371.755         261          36   
46    6        103.447374    85.160   115.065         257          42   
17    7        101.107711    85.045   115.125         249          38   
39    9        238.860717   216.380   265.890         244          24   
12    3        231.004393   212.145   272.820         239          31   
39    10       338.830696   278.600   365.370         237          26   
47    6        103.874677    84.080   115.270         232          39   
13    7         99.381317    84.565   115.270         224          39   
9     8        230.532434   214.795   268.605         189          27   
43    6         99.273901    85.160   114.845         182          32   
49    12       436.214222   391.245   497.500         180          28   
41    6         98.846798    85.895   115.275         178          30   
48    3        229.887286   212.675   273.270         175          28   
13    9        241.409128   216.115   273.905         172          25   
19    7        225.420382   212.530   255.550         170          43   
10    9        334.852852   277.000   372.665         149          31   
4     7        102.209388    85.025   115.265         147          20   
47    9        235.440205   212.045   272.355         146          27   
19    8        341.003938   329.320   357.895         146          41   
9     9        342.541414   299.680   359.135         145          27   
30    3        236.553531   213.120   264.270         143          14   
23    10       233.913794   216.110   261.250         141          21   
20    7        223.689964   213.075   246.400         140          24   
39    6         99.975827    84.730   113.490         139          16   
45    8        232.137689   213.345   269.120         132          27   
37    7        245.358385   217.070   262.990         130          13   
46    10       339.770508   276.015   366.640         128          25   
18    7        100.171600    84.530   115.180         125          18   
43    7        145.748185   130.885   162.005         124          21   
5     5        248.897521   215.225   273.955         119          12   
6     6        218.651144   213.095   224.750         118          26   
4     8        226.939612   215.880   246.005         116          17   
12    4        338.105965   277.010   364.165         114          22   
28    8        225.054670   212.505   234.630         106          16   
24    9        322.231373   275.845   359.900         102          18   
5     3         98.364406    85.250   114.705         101          12   
47    10       338.925750   279.510   368.145         100          15   
16    6        102.205400    85.015   114.890         100          10   

               band_min  band_max  band_mode    width  
topic cluster                                          
49    10              6         6        6.0   61.980  
43    9               6         6        6.0   54.0

### Check to see there are no clusters spanning bands

In [91]:
topic_cluster_stats.query('band_max - band_min > 1 and cluster != -1')\
    .sort_values(['width', 'count_freq'], ascending=False)\
    .head(30)

,,mean_freq,min_freq,max_freq,count_freq,count_proj,band_min,band_max,band_mode,width
topic,cluster,,,,,,,,,


### Inspect an individual topic's clusters

In [92]:
topic_cluster_stats.loc[0]

,mean_freq,min_freq,max_freq,count_freq,count_proj,band_min,band_max,band_mode,width
cluster,,,,,,,,,
-1,340.390000,338.130,342.875,4,2,7,7,7.0,4.745
0,348.951250,345.945,351.930,4,2,7,7,7.0,5.985
1,178.390000,175.765,182.000,3,1,5,5,5.0,6.235
2,165.958333,164.165,168.005,3,1,5,5,5.0,3.840
3,329.952500,329.325,330.580,2,1,7,7,7.0,1.255
4,145.757500,144.915,146.600,2,1,4,4,4.0,1.685
5,250.560909,244.245,254.530,11,4,6,6,6.0,10.285
6,134.082500,133.295,134.870,2,1,4,4,4.0,1.575
7,102.378125,86.845,110.895,24,3,3,3,3.0,24.050


### Histogram of topic clusters

**Compare this to the scatter plot above. These two charts in tandem are good. Maybe we can combine them somehow**

Hover info needs work

In [93]:
import plotly.graph_objects as go
def plot_topic_clusters(tc_frame:pd.DataFrame, topic:int):
    figure = go.Figure()
    figure.add_trace(
        go.Bar(
            x=tc_frame.query(f'topic== {topic} and cluster != -1').mean_freq,
            y=tc_frame.query(f'topic== {topic} and cluster != -1').count_freq,
            #name=dict(color=tc_frame.query(f'topic== {topic} and cluster != -1').index.get_level_values(level=1)),
            width=tc_frame.query(f'topic== {topic} and cluster != -1').width.to_list()
            # hoverinfo=(
            #     tc_frame.query(f'topic== {topic} and cluster != -1').min_freq,
            #     tc_frame.query(f'topic== {topic} and cluster != -1').max_freq
            # )
        )
    )
    figure.update_layout(
    title=(f'Areas of Interest for Topic {topic}'),
    xaxis_title='Frequency (GHz)',
    yaxis_title='Count of Measurements',
    )
    figure.show()

In [94]:
plot_topic_clusters(topic_cluster_stats, 25)